In [1]:
# info for eliminated proteins
excluded_protein_ids = []
excluded_gene_tags = [] # excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0])
removed_stages = []
excluded_reasons = []
excluded_evidences = []

# Proteome Creation

In [3]:
# assemble proteome data
import os
import pandas as pd
from Bio import SeqIO

gbff_file = "ATCC17978_mff/ncbi_dataset/data/GCF_001077675.1/genomic.gbff"

# Lists to store extracted data
protein_ids = []
locus_tags = []
protein_names = []
lengths = []
functions = []


# Read the GenBank file
for record in SeqIO.parse(gbff_file, "genbank"):
    for feature in record.features:
        # We only want protein-coding features (CDS) that have a protein ID
        if feature.type == "CDS" and "protein_id" in feature.qualifiers:
            
            # 1. Protein ID
            pid = feature.qualifiers["protein_id"][0]
            
            # 2. Gene/Locus Tag (falls back to locus_tag if 'gene' is missing)
            ltag = feature.qualifiers.get("locus_tag", [""])[0]
            
            # 3. Protein Name (stored under 'product' in NCBI files)
            pname = feature.qualifiers.get("product", ["Unknown Protein"])[0]
            
            # 4. Length (aa) - calculated directly from the translation sequence length
            if "translation" in feature.qualifiers:
                aa_len = len(feature.qualifiers["translation"][0])
            else:
                # Fallback calculation from nucleotide coordinates if translation text is missing
                aa_len = len(feature.location) // 3 
                
            # 5. Function (Notes or structural functions annotated by NCBI)
            func = feature.qualifiers.get("function", feature.qualifiers.get("note", ["Hypothetical"]))[0]
            
            # Append everything to our lists
            protein_ids.append(pid)
            locus_tags.append(ltag)
            protein_names.append(pname)
            lengths.append(aa_len)
            functions.append(func)

# Create dataframe
df = pd.DataFrame({
    "Protein ID": protein_ids,
    "Gene/Locus Tag": locus_tags,
    "Protein Name": protein_names,
    "Length (aa)": lengths,
    "Function": functions
})

# Save to an Excel file
output_excel = "Complete_A.baumanni_Proteome.xlsx"
proteome_loc = '/Users/tanmayi_akasapu/Downloads/Research Labs/BioTechTrek/'+output_excel
df.to_excel(proteome_loc, index=False)


# Blastp Human Homolog Filtering

In [6]:
# run blastp to find human homologs
from pathlib import Path

query = "ATCC17978_mff/ncbi_dataset/data/GCF_001077675.1/protein.faa"

reference = 'human_reference_proteome.fasta'

!makeblastdb -in '{reference}' -dbtype prot -out ref_proteome_db

db = "ref_proteome_db"

output = "blastp_homology_results.tsv"

!blastp \
    -query "{query}" \
    -db "{db}" \
    -out "{output}" \
    -evalue 1e-5 \
    -outfmt "6 qseqid sseqid pident length qlen slen qcovs evalue bitscore" \
    -num_threads 4




Building a new DB, current time: 10/24/2026 21:23:08
New DB name:   /Users/tanmayi_akasapu/Downloads/Research Labs/BioTechTrek/A. baumannii Subtractive Proteomics/ref_proteome_db
New DB title:  human_reference_proteome.fasta
Sequence type: Protein
Deleted existing Protein BLAST database named /Users/tanmayi_akasapu/Downloads/Research Labs/BioTechTrek/A. baumannii Subtractive Proteomics/ref_proteome_db
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 147520 sequences in 1.32615 seconds.


BLAST Database error: No alias or index file found for protein database [/Users/tanmayi_akasapu/Downloads/Research] in search path [/Users/tanmayi_akasapu/Downloads/Research Labs/BioTechTrek/A. baumannii Subtractive Proteomics::]. Please verify the spelling of the BLAST database and its molecule type.
^C


In [133]:
# Isolate proteins than satisfy homolog conditions
import pandas as pd

blastp_df = pd.read_excel('blastp_homology_results.xlsx')
decisions = []

for idx, row in blastp_df.iterrows():
    
    has_homolog = (row['E-Value'] <= 1e-5) and (row['% Identity'] >= 30) and (row['Query Coverage'] >= 70)
    
    if has_homolog:
        decisions.append('Remove')
        excluded_protein_ids.append(row['Query Protein'])
        excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Query Protein'], 'Gene/Locus Tag'].values[0])
        removed_stages.append('BLASTp')
        excluded_reasons.append('Significant human homolog similarity')
        excluded_evidences.append(f'E-value = {row['E-Value']}')
        
    else:
        decisions.append('Retain')



blastp_df['Decision'] = decisions

nonhomologous_proteins_df = blastp_df[blastp_df['Decision'] == 'Retain']
nonhomologous_proteins_df = nonhomologous_proteins_df.drop_duplicates(subset=['Query Protein'])
display(nonhomologous_proteins_df)
nonhomologous_proteins_df.to_excel('human_homology_filter.xlsx', index=False)

,Query Protein,Best Human Hit,% Identity,Alignment Length,Unnamed: 4,Unnamed: 5,Query Coverage,E-Value,Bit Score,Decision
0,WP_000004971.1,tr|A0ACI8R726|A0ACI8R726_HUMAN,21.698,424,486,459,81,2.610000e-08,58.2,Retain
97,WP_000005756.1,tr|A0ACI8RR80|A0ACI8RR80_HUMAN,42.122,311,495,371,62,1.420000e-66,220.0,Retain
267,WP_000006962.1,sp|Q96CD2|COAC_HUMAN,36.111,180,417,204,42,1.310000e-20,91.3,Retain
269,WP_000007337.1,tr|A0A499FJN5|A0A499FJN5_HUMAN,25.764,229,338,351,65,1.820000e-16,81.3,Retain
277,WP_000008535.1,tr|C9IZ01|C9IZ01_HUMAN,26.819,481,530,591,85,2.610000e-46,172.0,Retain
...,...,...,...,...,...,...,...,...,...,...
58355,WP_170825892.1,sp|Q0GE19|NTCP7_HUMAN,27.187,320,323,340,97,1.200000e-25,106.0,Retain
58360,WP_170825897.1,tr|A0A7P0Z4L8|A0A7P0Z4L8_HUMAN,20.428,607,628,638,87,3.760000e-15,81.3,Retain
58398,WP_170825911.1,sp|P45954|ACDSB_HUMAN,28.809,361,427,432,81,7.690000e-37,141.0,Retain
58531,WP_179994561.1,tr|F8VXU8|F8VXU8_HUMAN,28.696,115,266,906,43,3.190000e-06,50.1,Retain


In [35]:
# delete from proteome
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

records = []
proteome_ids = []
for record in SeqIO.parse(query, "fasta"):
    records.append(record)
    proteome_ids.append(record.id)

nonhomologs_list = nonhomologous_proteins_df['Query Protein'].tolist()

for h in nonhomologs_list:
    index = proteome_ids.index(h)
    proteome_ids.pop(index)
    records.pop(index)

SeqIO.write(records, "homolog_removed_proteome.fasta", "fasta")

ValueError: 'WP_000004971.1' is not in list

# DEG Essentiality Analysis

In [7]:
# A. Baumanni DEG isolation from DEG10

deg_file = 'DEG10.aa'

records = list(SeqIO.parse(deg_file, "fasta"))

deg10_protein_data = []

for record in SeqIO.parse(deg_file, "fasta"):
    deg10_protein_data.append({
        "DEG_ID": record.id,
        "Description": record.description,
        "Sequence": str(record.seq),
        "Length": len(record.seq)
    })

deg10_df = pd.DataFrame(deg10_protein_data)
display(deg10_df)

ab_deg = []

for idx, row in deg10_df.iterrows():
    org_id = row['DEG_ID'][:7]

    if (org_id == 'DEG1043') or (org_id == 'DEG1044'):
        ab_deg.append(row)

ab_deg_df = pd.DataFrame(ab_deg)
display(ab_deg_df)

# save to blastp-friendly file
deg_records = [
    SeqRecord(
        Seq(str(row["Sequence"])),
        id=str(row["DEG_ID"]),
        description=""
    )
    for _, row in ab_deg_df.iterrows()
]

SeqIO.write(
    deg_records,
    "AB_DEG_essential.fasta",
    "fasta"
)

print("Essential proteins saved:", len(deg_records))


,DEG_ID,Description,Sequence,Length
0,DEG10010001,DEG10010001,MENILDLWNQALAQIEKKLSKPSFETWMKSTKAHSLQGDTLTITAP...,446
1,DEG10010002,DEG10010002,MKFTIQKDRLVESVQDVLKAVSSRTTIPILTGIKIVASDDGVSFTG...,378
2,DEG10010003,DEG10010003,MEQQQNSYDENQIQVLEGLEAVRKRPGMYIGSTNSKGLHHLVWEIV...,638
3,DEG10010004,DEG10010004,MSEQNTPQVREINISQEMRTSFLDYAMSVIVSRALPDVRDGLKPVH...,821
4,DEG10010005,DEG10010005,MWESKFSKEGLTFDDVLLVPAKSEVLPRDVDLSVELTKTLKLNIPV...,488
...,...,...,...,...
26614,DEG10660379,DEG10660379,FMAFLFSRGLENTAVTIHALAARLI,25
26615,DEG10660380,DEG10660380,MPVLYGAKPAQPLLGKLEAELGFVLPADYKDFLKKWNGIFIGGADY...,174
26616,DEG10660381,DEG10660381,MAQAIDETSRIADLKIRAVPSEGQYLNTAQIARDVAELFATDIEGS...,228
26617,DEG10660382,DEG10660382,MVITIGAEKGGVAKTRLATHIAALAAAEDVDVVLLDTDKQGSAMSW...,223


,DEG_ID,Description,Sequence,Length
16067,DEG10430001,DEG10430001,MRQVEILVDSRPGSILSSSEQPATTTAALQTAPIPQPAKVKREPEP...,398
16068,DEG10430002,DEG10430002,MEICKSLPTAALIDLQITEDQRCILKSGNSRFVLGTLPAEDYPLLT...,305
16069,DEG10430003,DEG10430003,MYIGDTDDGTGLHHMVFEVVDNAIDEALAGHCDEIIVTIHEDESVS...,784
16070,DEG10430004,DEG10430004,MLRVYVTGGGCSGFSYGFNFAESVNEDDAEFVNGDVKMLVDSLSYQ...,83
16071,DEG10430005,DEG10430005,MIGDPTGKSATRPPLSREQVEANAKTYQEQVFKILDPNKTKVRFNS...,324
...,...,...,...,...
16676,DEG10440152,DEG10440152,MQVLSVHSLQILDVGQAVIHNQLTLGIVVASENETATALAMKEILI...,386
16677,DEG10440153,DEG10440153,MYPAPTVDQLAIDHAPKFENKRGNRFALPRPEPLQTDTTADASAQT...,147
16678,DEG10440154,DEG10440154,MEKLAEAGIETHFEKLLSPTEVLVKKLQMIPVECVIRNYAAGSLCR...,182
16679,DEG10440155,DEG10440155,MVQQIDAPLREDVRLLGNLLGETLKQHAGQELFNQIEQIRALAKGA...,894


Essential proteins saved: 614


In [8]:
# run blastp
import subprocess

query = Path('homolog_removed_proteome.fasta')

db = "ATCC17978_nonhomologous_db"

subprocess.run([
    "makeblastdb",
    "-in", str(query),
    "-dbtype", "prot",
    "-out", db
], check=True)

subprocess.run([
    "blastp",
    "-query", "AB_DEG_essential.fasta",
    "-db", db,
    "-out", "DEG_vs_proteome.tsv",
    "-evalue", "1e-5",
    "-outfmt",
    "6 qseqid sseqid pident length qlen slen "
    "qcovhsp evalue bitscore",
    "-num_threads", "4"
], check=True)

NameError: name 'subprocess' is not defined

In [134]:
# DEG Analysis

deg_blast_df = pd.read_excel('DEG_vs_proteome.xlsx')

deg_blast_df["Subject Coverage"] = (
    deg_blast_df["Alignment Length"]
    / deg_blast_df["Subject Length"]
    * 100
)

essential_status = []

for idx, row in deg_blast_df.iterrows():
    supported = (row["% Identity"] >= 90) & (row["Query Coverage"] >= 90) & (row["Subject Coverage"] >= 90) & (row["E-Value"] <= 1e-5)
    
    if supported:
        essential_status.append('Essential')
    else:
        essential_status.append('No supporting evidence')
        excluded_protein_ids.append(row['Nonhomologous Protein Hit'])
        excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Nonhomologous Protein Hit'], 'Gene/Locus Tag'].values[0])
        removed_stages.append('DEG')
        excluded_reasons.append('No supporting essentiality evidence')
        excluded_evidences.append(f'E-value = {row["E-Value"]}')


deg_blast_df['Evidence'] = essential_status
deg_blast_df = deg_blast_df.drop(columns=['% Identity', 'Alignment Length', 'Query Length', 'Subject Length', 'Query Coverage', 'E-Value', 'Bit Score'])
gene_tags = []

proteome_df = pd.read_excel('Complete_A.baumanni_Proteome.xlsx')

for idx, row in deg_blast_df.iterrows():
    protein_id = row['Nonhomologous Protein Hit']

    gene_tags.append(proteome_df.loc[proteome_df['Protein ID'] == protein_id, 'Gene/Locus Tag'].values[0])


temp = {
    'Protein ID': deg_blast_df['Nonhomologous Protein Hit'],
    'Gene/Locus Tag': gene_tags,
    'DEG Match': deg_blast_df['DEG Protein'],
    'DEG Reference Organism': 'A. baumannii ATCC 17978',
    'Essentiality Evidence': deg_blast_df['Evidence'],
    'Evidence Type': 'Experimental'
}

deg_analysis_df = pd.DataFrame(temp)

deg_decisions = []

for _,row in deg_analysis_df.iterrows():
    has_evidence = row['Essentiality Evidence'] == 'Essential'

    if has_evidence:
        deg_decisions.append('Retain')

    else:
        deg_decisions.append('Secondary candidate')
        
        

deg_analysis_df['Decision'] = deg_decisions

display(deg_analysis_df)
deg_analysis_df.to_excel('DEG_results.xlsx', index=False)


,Protein ID,Gene/Locus Tag,DEG Match,DEG Reference Organism,Essentiality Evidence,Evidence Type,Decision
0,WP_000964767.1,ACX60_RS00005,DEG10430001,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
1,WP_001237350.1,ACX60_RS00010,DEG10430002,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
2,WP_001133879.1,ACX60_RS17980,DEG10430006,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
3,WP_169538965.1,ACX60_RS17845,DEG10430010,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
4,WP_001075972.1,ACX60_RS17830,DEG10430011,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
...,...,...,...,...,...,...,...
573,WP_000250585.1,ACX60_RS00155,DEG10440155,A. baumannii ATCC 17978,Essential,Experimental,Retain
574,WP_001113592.1,ACX60_RS00130,DEG10440156,A. baumannii ATCC 17978,Essential,Experimental,Retain
575,WP_001060808.1,ACX60_RS08620,DEG10440156,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate
576,WP_000999636.1,ACX60_RS08870,DEG10440156,A. baumannii ATCC 17978,No supporting evidence,Experimental,Secondary candidate


# BlastKOALA/KEGG Pathway Analysis

In [46]:
# prepare BlastKOALA input
kegg_input = deg_analysis_df[deg_analysis_df['Decision'] == 'Retain']
display(kegg_input)

essential_nonhomologs = kegg_input['Protein ID'].tolist()

records = []
for record in SeqIO.parse('ATCC17978_mff/ncbi_dataset/data/GCF_001077675.1/protein.faa', "fasta"):
    records.append(record)

kegg_fasta = []
for record in records:
    if record.id in essential_nonhomologs:
        kegg_fasta.append(record)

SeqIO.write(kegg_fasta, "essential_nonhomologs.fasta", "fasta")

#==========RUN ON BLASTKOALA==========#


,Protein ID,Gene/Locus Tag,DEG Match,DEG Reference Organism,Essentiality Evidence,Evidence Type,Decision
6,WP_000872590.1,ACX60_RS17820,DEG10430013,A. baumannii ATCC 17978,Essential,Experimental,Retain
7,WP_000712680.1,ACX60_RS17810,DEG10430015,A. baumannii ATCC 17978,Essential,Experimental,Retain
8,WP_000011809.1,ACX60_RS17805,DEG10430016,A. baumannii ATCC 17978,Essential,Experimental,Retain
13,WP_009519256.1,ACX60_RS17795,DEG10430018,A. baumannii ATCC 17978,Essential,Experimental,Retain
17,WP_000697209.1,ACX60_RS17770,DEG10430020,A. baumannii ATCC 17978,Essential,Experimental,Retain
...,...,...,...,...,...,...,...
562,WP_001139944.1,ACX60_RS01565,DEG10440144,A. baumannii ATCC 17978,Essential,Experimental,Retain
566,WP_000905538.1,ACX60_RS01290,DEG10440148,A. baumannii ATCC 17978,Essential,Experimental,Retain
570,WP_000211305.1,ACX60_RS00740,DEG10440150,A. baumannii ATCC 17978,Essential,Experimental,Retain
573,WP_000250585.1,ACX60_RS00155,DEG10440155,A. baumannii ATCC 17978,Essential,Experimental,Retain


145

In [138]:
kegg_results = pd.read_excel('A.baumannii_KEGG_IDs.xlsx')

display(kegg_results)

gene_tags = []
for _,row in kegg_results.iterrows():
    gene_tags.append(kegg_input.loc[kegg_input['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0])

url = "https://rest.kegg.jp/link/pathway/ko"

response = requests.get(url, timeout=60)
response.raise_for_status()

ko_pathway_df = pd.read_csv(
    StringIO(response.text),
    sep="\t",
    header=None,
    names=["KO", "Pathway ID"]
)

# Clean IDs
ko_pathway_df["KO"] = (
    ko_pathway_df["KO"]
    .str.replace("ko:", "", regex=False)
)

ko_pathway_df["Pathway ID"] = (
    ko_pathway_df["Pathway ID"]
    .str.replace("path:", "", regex=False)
)

url = "https://rest.kegg.jp/list/pathway"

response = requests.get(url, timeout=60)
response.raise_for_status()

pathway_desc_df = pd.read_csv(
    StringIO(response.text),
    sep="\t",
    header=None,
    names=["Pathway ID", "Pathway Description"]
)

pathway_desc_df["Pathway ID"] = (
    pathway_desc_df["Pathway ID"]
    .str.replace("path:", "", regex=False)
)

display(ko_pathway_df)
display(pathway_desc_df)

,Protein ID,KO
0,WP_000003406.1,K04085
1,WP_000007423.1,K02492
2,WP_000009640.1,K00641
3,WP_000011809.1,K13018
4,WP_000013434.1,K00748
...,...,...
140,WP_004748097.1,NaN
141,WP_009517740.1,NaN
142,WP_009519256.1,NaN
143,WP_074031668.1,K04075


,KO,Pathway ID
0,K00001,map00010
1,K00001,ko00010
2,K00002,map00010
3,K00002,ko00010
4,K00016,map00010
...,...,...
98137,K20607,ko05418
98138,K21278,map05418
98139,K21278,ko05418
98140,K23790,map05418


,Pathway ID,Pathway Description
0,map01100,Metabolic pathways
1,map01110,Biosynthesis of secondary metabolites
2,map01120,Microbial metabolism in diverse environments
3,map01200,Carbon metabolism
4,map01210,2-Oxocarboxylic acid metabolism
...,...,...
582,map07035,Prostaglandins
583,map07110,Benzoic acid family
584,map07112,"1,2-Diphenyl substitution family"
585,map07114,Naphthalene family


In [139]:
# find associated pathways
pathways = []
descriptions = []

for _, row in kegg_results.iterrows():

    ko = row["KO"]

    if pd.notna(ko) and str(ko).strip() not in ["", "NaN", "nan", "-"]:

        matching_paths = ko_pathway_df.loc[
            ko_pathway_df["KO"] == str(ko).strip(),
            "Pathway ID"
        ]

        if not matching_paths.empty:

            path = matching_paths.iloc[0]

            matching_desc = pathway_desc_df.loc[
                pathway_desc_df["Pathway ID"] == path,
                "Pathway Description"
            ]

            description = (
                matching_desc.iloc[0]
                if not matching_desc.empty
                else "No Pathway Description"
            )

            pathways.append(path)
            descriptions.append(description)

        else:
            pathways.append("No KEGG Pathway Identified")
            descriptions.append("No KEGG Pathway Identified")

    else:
        pathways.append("No KEGG Pathway Identified")
        descriptions.append("No KEGG Pathway Identified")

kegg_results = {
    'Protein ID': kegg_results['Protein ID'],
    'Gene/Locus Tag': gene_tags,
    'KO': kegg_results['KO'],
    'Pathway ID': pathways,
    'Pathway Description': descriptions
}



kegg_results_df = pd.DataFrame(kegg_results)
display(kegg_results_df)

# record eliminated proteins
for _,row in kegg_results_df.iterrows():
    if row['Pathway ID'] == 'No KEGG Pathway Identified':
        excluded_protein_ids.append(row['Protein ID'])
        excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0])
        removed_stages.append('KEGG')
        excluded_reasons.append('No KEGG Pathway Identified')
        excluded_evidences.append('Unsuccessful KEGG search')

kegg_results_df.to_excel('KEGG_results.xlsx', index=False)

,Protein ID,Gene/Locus Tag,KO,Pathway ID,Pathway Description
0,WP_000003406.1,ACX60_RS16225,K04085,map04122,Sulfur relay system
1,WP_000007423.1,ACX60_RS14180,K02492,map00860,Porphyrin metabolism
2,WP_000009640.1,ACX60_RS15740,K00641,map00270,Cysteine and methionine metabolism
3,WP_000011809.1,ACX60_RS17805,K13018,map00541,Biosynthesis of various nucleotide sugars
4,WP_000013434.1,ACX60_RS09595,K00748,map00540,Lipopolysaccharide biosynthesis
...,...,...,...,...,...
140,WP_004748097.1,ACX60_RS13920,NaN,No KEGG Pathway Identified,No KEGG Pathway Identified
141,WP_009517740.1,ACX60_RS02460,NaN,No KEGG Pathway Identified,No KEGG Pathway Identified
142,WP_009519256.1,ACX60_RS17795,NaN,No KEGG Pathway Identified,No KEGG Pathway Identified
143,WP_074031668.1,ACX60_RS15075,K04075,No KEGG Pathway Identified,No KEGG Pathway Identified


# PSORTb Localization Analysis

In [62]:
# set up psortb input
found_pathways = kegg_results_df[kegg_results_df['Pathway ID'] != "No KEGG Pathway Identified"]
pathway_identified_proteins = found_pathways['Protein ID'].tolist()

display(found_pathways)

psortb_input = []
for record in records:
    if record.id in pathway_identified_proteins:
        psortb_input.append(record)

SeqIO.write(psortb_input, "pathways_identified.fasta", "fasta")

#==========RUN FILE IN PSORTb==========#

,Protein ID,Gene/Locus Tag,KO,Pathway ID,Pathway Description
0,WP_000003406.1,ACX60_RS16225,K04085,map04122,Sulfur relay system
1,WP_000007423.1,ACX60_RS14180,K02492,map00860,Porphyrin metabolism
2,WP_000009640.1,ACX60_RS15740,K00641,map00270,Cysteine and methionine metabolism
3,WP_000011809.1,ACX60_RS17805,K13018,map00541,Biosynthesis of various nucleotide sugars
4,WP_000013434.1,ACX60_RS09595,K00748,map00540,Lipopolysaccharide biosynthesis
...,...,...,...,...,...
132,WP_001291845.1,ACX60_RS02140,K02961,map03010,Ribosome
133,WP_001982638.1,ACX60_RS02120,K02890,map03010,Ribosome
134,WP_001982642.1,ACX60_RS02095,K02906,map03010,Ribosome
136,WP_001984863.1,ACX60_RS03965,K00241,map00020,Citrate cycle (TCA cycle)


91

In [140]:
# analyze psortb results
psortb_results = pd.read_excel('PSORTb results.xlsx')

protein_ids = []
protein_names = []
interpretations = []

for _,row in psortb_results.iterrows():
    cleaned_seqID = ''
    if 'MULTISPECIES: ' in row['SeqID']:
        cleaned_seqID = row['SeqID'].replace('MULTISPECIES: ', '')
    else:
        cleaned_seqID = row['SeqID']
    
    protein_ids.append(cleaned_seqID[:14])
    protein_names.append(cleaned_seqID[15:])

    if row['Localization'] == 'Unknown':
        interpretations.append('No PSORTb localization found')
    elif row['Localization'] == 'Cytoplasmic':
        interpretations.append('Intracellular')
    elif (row['Localization'] == 'CytoplasmicMembrane') or (row['Localization'] == 'OuterMembrane') or (row['Localization'] == 'Periplasmic'):
        interpretations.append('Membrane associated')
    elif row['Localization'] == 'Extracellular':
        interpretations.append('Extracellular')

psortb_results = {
    'Protein ID': protein_ids,
    'Protein': protein_names,
    'Localization': psortb_results['Localization'],
    'PSORTb Score': psortb_results['Score'],
    'Interpretation': interpretations
}

psortb_results_df = pd.DataFrame(psortb_results)
display(psortb_results_df)

for _,row in psortb_results_df.iterrows():
    if row['Interpretation'] != 'Membrane associated':
        excluded_protein_ids.append(row['Protein ID'])
        excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0])
        removed_stages.append('PSORTb')
        excluded_reasons.append('Not membrane associated')
        excluded_evidences.append(f'PSORTb localization = {row['Localization']}')

psortb_results_df.to_excel('PSORTb_results.xlsx',index=False)

,Protein ID,Protein,Localization,PSORTb Score,Interpretation
0,WP_000003406.1,sulfurtransferase TusA [Gammaproteobacteria],Cytoplasmic,9.26,Intracellular
1,WP_000007423.1,glutamyl-tRNA reductase [Acinetobacter],Cytoplasmic,9.97,Intracellular
2,WP_000009640.1,homoserine O-succinyltransferase MetX [Acineto...,Cytoplasmic,9.97,Intracellular
3,WP_000011809.1,"UDP-2-acetamido-3-amino-2,3-dideoxy-D-glucuron...",Cytoplasmic,9.26,Intracellular
4,WP_000013434.1,lipid-A-disaccharide synthase [Acinetobacter],Cytoplasmic,9.97,Intracellular
...,...,...,...,...,...
86,WP_001291845.1,30S ribosomal protein S17 [Gammaproteobacteria],Cytoplasmic,9.26,Intracellular
87,WP_001982638.1,50S ribosomal protein L22 [Bacteria],Cytoplasmic,9.26,Intracellular
88,WP_001982642.1,50S ribosomal protein L3 [Gammaproteobacteria],Cytoplasmic,9.97,Intracellular
89,WP_001984863.1,"succinate dehydrogenase, cytochrome b556 subun...",CytoplasmicMembrane,10.00,Membrane associated


# DrugBank Druggability Analysis

In [94]:
# isolate membrane proteins and search each protein name in DrugBank
membrane_proteins = psortb_results_df[psortb_results_df['Interpretation'] == 'Membrane associated']

gene_tags = []
for _,row in membrane_proteins.iterrows():
    gene = kegg_input.loc[kegg_input['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0]
    gene_tags.append(gene)
    print(gene)


ACX60_RS01340
ACX60_RS16685
ACX60_RS15245
ACX60_RS05960
ACX60_RS01490
ACX60_RS00450
ACX60_RS17335
ACX60_RS17820
ACX60_RS01985
ACX60_RS02900
ACX60_RS14615
ACX60_RS03965
ACX60_RS02195


/var/folders/3r/x_yfn5cj0n1b3r2g20_phb800000gn/T/ipykernel_871/4126284363.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  membrane_proteins['Gene/Locus Tag'] = gene_tags


,Protein ID,Protein,Localization,PSORTb Score,Interpretation,Gene/Locus Tag
6,WP_000051217.1,ATP-binding protein [Acinetobacter],CytoplasmicMembrane,9.99,Membrane associated,ACX60_RS01340
8,WP_000063880.1,preprotein translocase subunit SecE [Gammaprot...,CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS16685
9,WP_000064536.1,GacS-like sensor histidine kinase [Acinetobact...,CytoplasmicMembrane,9.99,Membrane associated,ACX60_RS15245
10,WP_000075224.1,rod shape-determining protein RodA [Acinetobac...,CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS05960
33,WP_000227936.1,penicillin-binding protein PBP3 [Gammaproteoba...,CytoplasmicMembrane,9.82,Membrane associated,ACX60_RS01490
41,WP_000380676.1,phosphatidylglycerophosphatase A family protei...,CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS00450
50,WP_000718586.1,F0F1 ATP synthase subunit A [Gammaproteobacteria],CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS17335
58,WP_000872590.1,polysaccharide biosynthesis/export family prot...,OuterMembrane,9.93,Membrane associated,ACX60_RS17820
68,WP_001091949.1,lipid asymmetry maintenance ABC transporter pe...,CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS01985
80,WP_001270222.1,protein translocase subunit SecD [Acinetobacter],CytoplasmicMembrane,10.00,Membrane associated,ACX60_RS02900


# STRING Protein Interaction Analysis

In [98]:
# extract RefSeq to Locus Tag mapping
string_input = pd.read_excel('Membrane_associated_proteins.xlsx')
display(string_input)

locus_formats = {}
for _,row in string_input.iterrows():
    locus_formats[row['Locus Tag Format']] = row['Protein ID']

,Protein ID,Protein,Known Target?,Drugs/Compounds,Evidence,Interpretation,Locus Tag Format
0,WP_000051217.1,ATP-binding protein [Acinetobacter],No,DB00626 Bacitracin,Potential homolog record,Potential target evidence,A1S_1167
1,WP_000063880.1,preprotein translocase subunit SecE [Gammaprot...,No,NaN,No record identified,Novel candidate,A1S_0753
2,WP_000064536.1,GacS-like sensor histidine kinase [Acinetobact...,No,NaN,No record identified,Novel candidate,A1S_0755
3,WP_000075224.1,rod shape-determining protein RodA [Acinetobac...,No,NaN,No record identified,Novel candidate,A1S_0757
4,WP_000227936.1,penicillin-binding protein PBP3 [Gammaproteoba...,No,DB00535 Cefdinir,Potential homolog record,Potential target evidence,A1S_0756
5,WP_000380676.1,phosphatidylglycerophosphatase A family protei...,No,NaN,No record identified,Novel candidate,A1S_0762
6,WP_000718586.1,F0F1 ATP synthase subunit A [Gammaproteobacteria],No,NaN,No record identified,Novel candidate,A1S_0758
7,WP_000872590.1,polysaccharide biosynthesis/export family prot...,No,NaN,No record identified,Novel candidate,A1S_0761
8,WP_001091949.1,lipid asymmetry maintenance ABC transporter pe...,No,NaN,No record identified,Novel candidate,A1S_0752
9,WP_001270222.1,protein translocase subunit SecD [Acinetobacter],No,NaN,No record identified,Novel candidate,A1S_0763


In [126]:
# condense STRING output
string_output = pd.read_csv('string_interactions_short_13_nodes.tsv', sep='\t')

candidates = []
interacting_proteins = []
scores = []
evidence_types = []
partner_functions = []

for _,row in string_output.iterrows():
    candidates.append(locus_formats[row['#node1']])
    interacting_proteins.append(locus_formats[row['node2']])
    scores.append(row['combined_score'])
    indiv_scores = string_output.iloc[_,4:11].tolist()
    highest_idx = indiv_scores.index(max(indiv_scores))

    if highest_idx == 0:
        evidence_types.append('Neighbor')
    elif highest_idx == 1:
        evidence_types.append('Gene Fusion')
    elif highest_idx == 2:
        evidence_types.append('Phylogenetic Co-occurrence')
    elif highest_idx == 3:
        evidence_types.append('Homology')
    elif highest_idx == 4:
        evidence_types.append('Co-expression')
    elif (highest_idx == 5) or (highest_idx == 6):
        evidence_types.append('Database/Experiment')

string_condensed_results = pd.DataFrame({
    'Candidate': candidates,
    'Interacting Protein': interacting_proteins,
    'Score': scores,
    'Evidence Type': evidence_types
})

display(string_condensed_results)
string_condensed_results.to_excel('STRING_results.xlsx',index=False)

,Candidate,Interacting Protein,Score,Evidence Type
0,WP_001091949.1,WP_000063880.1,0.999,Database/Experiment
1,WP_001091949.1,WP_000064536.1,0.999,Database/Experiment
2,WP_001091949.1,WP_000227936.1,0.999,Database/Experiment
3,WP_001091949.1,WP_000075224.1,0.999,Database/Experiment
4,WP_001091949.1,WP_000718586.1,0.999,Database/Experiment
...,...,...,...,...
61,WP_000872590.1,WP_001270222.1,0.999,Database/Experiment
62,WP_000872590.1,WP_002000592.1,0.999,Database/Experiment
63,WP_000380676.1,WP_001270222.1,0.999,Database/Experiment
64,WP_000380676.1,WP_002000592.1,0.999,Database/Experiment


In [111]:
# find STRING degree

string_input_proteins = string_input['Protein ID'].tolist()
occurences = string_condensed_results['Candidate'].tolist()


degree = []
for protein in string_input_proteins:
    degree.append(occurences.count(protein))

string_degrees = pd.DataFrame({
    'Protein ID': string_input_proteins,
    'STRING Degree': degree
})

display(string_degrees)

,Protein ID,STRING Degree
0,WP_000051217.1,0
1,WP_000063880.1,10
2,WP_000064536.1,9
3,WP_000075224.1,7
4,WP_000227936.1,8
5,WP_000380676.1,2
6,WP_000718586.1,6
7,WP_000872590.1,3
8,WP_001091949.1,11
9,WP_001270222.1,1


# Final Candidate Selection

In [141]:
high_string_deg = string_degrees[string_degrees['STRING Degree'] >= 8]

for _,row in string_degrees.iterrows():
    if row['STRING Degree'] < 8:
        excluded_protein_ids.append(row['Protein ID'])
        excluded_gene_tags.append(df.loc[df['Protein ID'] == row['Protein ID'], 'Gene/Locus Tag'].values[0])
        removed_stages.append('STRING')
        excluded_reasons.append('Low STRING degree')
        excluded_evidences.append(f'STRING degree = {row['STRING Degree']}')

final_protein_ids = high_string_deg['Protein ID'].tolist()
gene_tags = []
funcs = []
lengths = []
kegg_pathways = []
drug_evidences = []


for protein in final_protein_ids:
    gene_tags.append(df.loc[df['Protein ID'] == protein, 'Gene/Locus Tag'].values[0])
    funcs.append(df.loc[df['Protein ID'] == protein, 'Function'].values[0])
    lengths.append(df.loc[df['Protein ID'] == protein, 'Length (aa)'].values[0])
    kegg_pathways.append(kegg_results_df.loc[kegg_results_df['Protein ID'] == protein, 'Pathway Description'].values[0])
    drug_evidences.append(string_input.loc[string_input['Protein ID'] == protein, 'Evidence'].values[0])

final_candidates_df = pd.DataFrame({
    'Protein ID': final_protein_ids,
    'Gene/Locus Tag': gene_tags,
    'Function': funcs,
    'Length': lengths,
    'Human Homology': 'No significant match', # all homologs filtered
    'Essentiality': 'Supported', # all nonessential proteins filtered
    'KEGG Pathway': kegg_pathways,
    'Localization': 'Membrane associated', # all non-membrane associated proteins filtered
    'Drug Evidence': drug_evidences,
    'STRING Degree': high_string_deg['STRING Degree'].tolist(),
    'Final Interpretation': 'Potential candidate'
})
display(final_candidates_df)
final_candidates_df.to_excel('final_candidate_table.xlsx', index=False)

,Protein ID,Gene/Locus Tag,Function,Length,Human Homology,Essentiality,KEGG Pathway,Localization,Drug Evidence,STRING Degree,Final Interpretation
0,WP_000063880.1,ACX60_RS16685,Derived by automated computational analysis us...,146,No significant match,Supported,Quorum sensing,Membrane associated,No record identified,10,Potential candidate
1,WP_000064536.1,ACX60_RS15245,Derived by automated computational analysis us...,935,No significant match,Supported,Two-component system,Membrane associated,No record identified,9,Potential candidate
2,WP_000227936.1,ACX60_RS01490,Derived by automated computational analysis us...,610,No significant match,Supported,Peptidoglycan biosynthesis,Membrane associated,Potential homolog record,8,Potential candidate
3,WP_001091949.1,ACX60_RS01985,Derived by automated computational analysis us...,258,No significant match,Supported,ABC transporters,Membrane associated,No record identified,11,Potential candidate


In [143]:
eliminated_proteins = pd.DataFrame({
    'Protein ID': excluded_protein_ids,
    'Gene/Locus Tag': excluded_gene_tags,
    'Stage Removed': removed_stages,
    'Reason': excluded_reasons,
    'Evidence': excluded_evidences
})
display(eliminated_proteins)
eliminated_proteins.to_excel('excluded_proteins.xlsx', index=False)

,Protein ID,Gene/Locus Tag,Stage Removed,Reason,Evidence
0,WP_000005756.1,ACX60_RS08560,BLASTp,Significant human homolog similarity,E-value = 6.18e-109
1,WP_000005756.1,ACX60_RS08560,BLASTp,Significant human homolog similarity,E-value = 2.66e-107
2,WP_000005756.1,ACX60_RS08560,BLASTp,Significant human homolog similarity,E-value = 2.96e-107
3,WP_000005756.1,ACX60_RS08560,BLASTp,Significant human homolog similarity,E-value = 5.3e-107
4,WP_000005756.1,ACX60_RS08560,BLASTp,Significant human homolog similarity,E-value = 6.82e-107
...,...,...,...,...,...
13358,WP_000872590.1,ACX60_RS17820,STRING,Low STRING degree,STRING degree = 3
13359,WP_001270222.1,ACX60_RS02900,STRING,Low STRING degree,STRING degree = 1
13360,WP_001274550.1,ACX60_RS14615,STRING,Low STRING degree,STRING degree = 5
13361,WP_001984863.1,ACX60_RS03965,STRING,Low STRING degree,STRING degree = 4
